# 화면을 다시 그려도 답과 출처가 섞이지 않는 챗봇

두 번째 질문의 검색이 실패했는데 화면에는 첫 번째 답의 출처가 남아 있다. 화면 상태와 그래프 상태의 경계를 시험한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 입력 한 번에 화면은 여러 번 그려질 수 있다

폼 제출, 버튼 클릭, 새 대화 선택은 서로 다른 사건이다. 화면이 다시 그려질 때마다 검색을 호출하면 같은 질문의 비용이 여러 번 발생한다.

질문·답·출처를 요청별 레코드 하나로 보관하면 실패한 다음 요청이 이전 출처를 물려받는 일을 막을 수 있다. thread_id는 서버 대화를 가리키고 화면에 표시할 레코드는 그 안의 개별 요청이다.

모델 연결 전에 두 개의 고정 응답을 넣어 렌더링 데이터를 검사한다. 이 시험이 통과하면 같은 코드를 Streamlit 폼 제출 처리에 연결해 브라우저에서 다시 확인한다.

## 빈 출처를 가진 두 번째 답 저장

출처 목록을 요청 결과에 포함시킨다. 응답이 빈 목록을 돌려주는 것도 이번 요청의 결과다.

In [1]:
history = {}
def save_turn(request_id, answer, sources):
    history[request_id] = {"answer":answer, "sources":list(sources)}

save_turn("q1", "연장 1회", ["R2"])
save_turn("q2", "확인할 근거가 없습니다.", [])
print(history["q2"])

{'answer': '확인할 근거가 없습니다.', 'sources': []}


q2를 읽었을 때 sources가 빈 목록이다. 화면에서 “마지막으로 남아 있던 출처 목록”을 별도 변수에서 가져오면 이 보장이 사라진다.

### 화면 갱신과 새 요청을 구별하기

표시 함수가 두 번 호출돼도 대화가 늘어나면 안 된다. 확인에 쓸 읽기 함수를 작성한다.

In [2]:
def visible_turns():
    # 기록을 추가하지 않고 현재 내용을 반환하세요.
    pass

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>def visible_turns():
    return list(history.items())
print(len(visible_turns()), len(visible_turns()), len(history))</code></pre><pre>2 2 2
</pre><p>두 번 그려도 저장된 요청은 두 개다. 실제 화면에서는 submit이 True인 경로에서만 실행 함수를 호출한다.</p></details>

## 늦게 끝난 요청이 새 대화에 도착하기

검색을 기다리는 사이 사용자가 새 대화를 클릭했다. 응답을 현재 열린 화면에 붙이기 전에 전송 당시의 대화 ID를 확인한다.

In [3]:
pending = {"r7":"chat-a"}
active = "chat-b"
def destination(request_id):
    return pending[request_id]

print(destination("r7") == active)

False


False는 요청이 실패했다는 뜻이 아니다. 완료된 응답의 목적지는 여전히 chat-a다. 화면을 전환했더라도 원래 대화에서 답을 볼 수 있어야 한다.

### 응답을 올바른 기록에 저장하기

도착한 응답을 요청 당시의 대화에 넣고 chat-b의 기록을 조회한다. 새 화면으로 넘어간 사용자가 이전 답을 보지 않는지 확인한다.

In [4]:
by_thread = {"chat-a":[], "chat-b":[]}

In [5]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>by_thread[destination(&quot;r7&quot;)].append({&quot;request&quot;:&quot;r7&quot;,&quot;text&quot;:&quot;연장 1회&quot;})
print(by_thread[&quot;chat-b&quot;], by_thread[&quot;chat-a&quot;])</code></pre><pre>[] [{&#x27;request&#x27;: &#x27;r7&#x27;, &#x27;text&#x27;: &#x27;연장 1회&#x27;}]
</pre><p>활성 화면은 비어 있고 이전 대화에만 답이 남는다. 실제 Streamlit에서는 이 값을 세션별로 보관하고 새로고침 시 재사용 정책을 정한다.</p></details>

## 다른 입력에 적용하기

Streamlit 폼에 같은 문장을 두 번 보낼 때와 “새 대화”를 누른 뒤 같은 문장을 보낼 때를 나눠 본다. 두 브라우저 세션을 열고 한쪽의 실패 메시지와 출처가 다른 쪽에 보이는지도 확인한다.

팀원에게 원문과 질문 한 건만 넘겨 주고 화면에서 인용 쪽까지 찾아가도록 한다. 필요한 버튼이나 다운로드가 없으면 화면 인수 조건에 추가한다.